# OncoGenerator quick start
Generate synthetic endometrial cancer clinical + protein data, look at the ground truth, and score a simple differential-expression test against it.

**All data are synthetic** - use them for method development and teaching, not for biological conclusions.

In [ ]:
# Run this cell in Google Colab (skip it if you already cloned the repo and are inside it)
!git clone https://github.com/NehaAr/OncoGenerator.git
%cd OncoGenerator
!pip -q install numpy pandas scipy scikit-learn

In [ ]:
import sys; sys.path.insert(0, ".")          # use the repo copy of the package
import numpy as np, pandas as pd
import oncogenerator as og

ds = og.generate_dataset(1000, seed=1, scenario="clean")   # change seed for an independent replicate
print(ds.clinical.shape, ds.proteins.shape)
ds.clinical.head()

In [ ]:
# Ground truth: how every protein was generated, and the true effect for each contrast
display(ds.protein_truth.head())
late = ds.contrast_truth[ds.contrast_truth.contrast == "Late_vs_Early_stage"]
print(late.is_nonzero_effect.value_counts().rename({True: "has effect", False: "true null"}))

In [ ]:
# Score a Welch t-test (late vs early stage) against the ground truth, with Benjamini-Hochberg FDR 5%
from scipy import stats
prot = ds.proteins.drop(columns="Patient_ID")
is_late = ds.clinical.Late_Stage.to_numpy() == 1
p = stats.ttest_ind(prot[is_late], prot[~is_late], equal_var=False).pvalue

def bh(p):
    o = np.argsort(p); r = p[o] * len(p) / (np.arange(len(p)) + 1)
    q = np.empty(len(p)); q[o] = np.minimum.accumulate(r[::-1])[::-1]; return q

hit = bh(p) < 0.05
truth = late.set_index("protein").loc[prot.columns, "is_nonzero_effect"].to_numpy()
print(f"found {hit.sum()} proteins; true positives {(hit & truth).sum()} of {truth.sum()}; false discoveries {(hit & ~truth).sum()}")

In [ ]:
# Try a harder scenario: more noise, or 20% low-value-biased missing data
og.SCENARIOS.keys()

## Next steps
- Command line: `python -m oncogenerator generate --n 10000 --seed 2 --scenario batch5 --out out/`
- More benchmarks: see `examples/` and `benchmarks/REPRODUCED.md`.
- *An Amazon Athena / SageMaker tutorial for the Parquet release is still to be written.*